### Import Data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

def dict_to_schema(col_type_dict):
    return StructType([
        StructField(col, typ, True) for col, typ in col_type_dict.items()
    ])

# Create a Spark session
spark = SparkSession.builder \
    .appName("Simple Example") \
    .getOrCreate()

# define the schema for each DataFrame if needed
member_schema_dict = {
    "ShopId": IntegerType(),
    "ShopMemberId": StringType(),
    "RegisterSourceTypeDef": StringType(),
    "RegisterDateTime": StringType(),  # Use StringType for datetime, then convert if needed
    "Gender": StringType(),
    "Birthday": StringType(),
    "APPRefereeId": IntegerType(),
    "APPRefereeLocationId": IntegerType(),
    "IsAppInstalled": StringType(),  # Boolean (True/False as strings in CSV)
    "IsEnableEmail": StringType(),
    "IsEnablePushNotification": StringType(),
    "IsEnableShortMessage": StringType(),
    "FirstAppOpenDateTime": StringType(),
    "LastAppOpenDateTime": StringType(),
    "MemberCardLevel": IntegerType(),
    "CountryAliasCode": StringType(),
}

order_tg_schema_dict = {
    "ShopId": IntegerType(),
    "ShopMemberId": StringType(),
    "TradesGroupCode": StringType(),
    "OrderDateTime": StringType(),
    "ChannelType": StringType(),
    "ChannelDetail": StringType(),
    "PaymentType": StringType(),
    "ShippingType": StringType(),
    "TsCount": IntegerType(),
    "Qty": IntegerType(),
    "TotalSalesAmount": DoubleType(),
    "TotalPrice": DoubleType(),
    "TotalDiscount": DoubleType(),
    "TotalPromotionDiscount": DoubleType(),
    "TotalCouponDiscount": DoubleType(),
    "TotalLoyaltyPointDiscount": DoubleType(),
    "StatusDef": StringType(),
}

order_ts_schema_dict = {
    "ShopId": IntegerType(),
    "ShopMemberId": StringType(),
    "TradesGroupCode": StringType(),
    "TradesSlaveCode": StringType(),
    "OrderDateTime": StringType(),
    "OrderFinishDateTime": StringType(),
    "ChannelType": StringType(),
    "ChannelDetail": StringType(),
    "PaymentType": StringType(),
    "ShippingType": StringType(),
    "OuterProductSkuCode": StringType(),
    "ProductSkuCode": StringType(),
    "SalePageId": StringType(),  # NOTE: StringType here!
    "Qty": IntegerType(),
    "UnitPrice": DoubleType(),
    "SubtotalPrice": DoubleType(),
    "SubtotalSalesAmount": DoubleType(),
    "SubtotalPromotionDiscount": DoubleType(),
    "SubtotalCouponDiscount": DoubleType(),
    "SubtotalLoyaltyPointDiscount": DoubleType(),
    "StatusDef": StringType(),
}

sale_page_schema_dict = {
    "ShopId": IntegerType(),
    "SalePageId": StringType(),  # Even if described as integer, better as String for joins
    "SalePageTitle": StringType(),
    "SaleProductDescShortContent": StringType(),
}

segment_schema_dict = {
    "ShopId": IntegerType(),
    "ShopMemberId": StringType(),
    "DataSourceDate": StringType(),  # Use StringType for date, convert later if needed
    "CategorySegment": StringType(),
}

schema_map = {
    'df_member': dict_to_schema(member_schema_dict),
    'df_order_tg': dict_to_schema(order_tg_schema_dict),
    'df_order_ts': dict_to_schema(order_ts_schema_dict),
    'df_sale_page': dict_to_schema(sale_page_schema_dict),
    'df_segment': dict_to_schema(segment_schema_dict),
}


base_path = '/Users/jasonshen/Desktop/我的大蟒蛇/BDA_2025/91APP_Dataset(會員&主單&子單&商品頁&標籤)'

file_map = {
    'df_member': 'Member.csv',
    'df_order_tg': 'Order_TG.csv',
    'df_order_ts': 'Order_TS.csv',
    'df_sale_page': 'SalePage.csv',
    'df_segment': 'Segment.csv',
}

# Read CSV files with specified schemas
for df_name, file_name in file_map.items():
    file_path = f"{base_path}/{file_name}"
    schema = schema_map[df_name]
    df = spark.read.csv(file_path, header=True, schema=schema)
    globals()[df_name] = df  # Store DataFrame in a variable with the name df_name

# Show the loaded data
for df_name in file_map.keys():
    print(f"{df_name}:")
    globals()[df_name].show(5)
    globals()[df_name].printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/25 14:32:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


df_member:
+------+--------------------+---------------------+--------------------+------+----------+------------+--------------------+--------------+-------------+------------------------+--------------------+--------------------+--------------------+---------------+----------------+
|ShopId|        ShopMemberId|RegisterSourceTypeDef|    RegisterDateTime|Gender|  Birthday|APPRefereeId|APPRefereeLocationId|IsAppInstalled|IsEnableEmail|IsEnablePushNotification|IsEnableShortMessage|FirstAppOpenDateTime| LastAppOpenDateTime|MemberCardLevel|CountryAliasCode|
+------+--------------------+---------------------+--------------------+------+----------+------------+--------------------+--------------+-------------+------------------------+--------------------+--------------------+--------------------+---------------+----------------+
|  NULL|SGE559ZPZi96JMssg...|                Store|2018-12-30 00:00:...|Female|1966-08-15|           0|                   0|          True|         True|           

25/05/25 14:32:35 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ShopId, ShopMemberId, TradesGroupCode, TradesSlaveCode, OrderDateTime, OrderFinishDateTime, ChannelType, ChannelDetail, PaymentType, ShippingType, OuterProductSkuCode, ProductSkuCode, SalePageId, Qty, UnitPrice, SubtotalSalesAmount, SubtotalPrice, SubtotalPromotionDiscount, SubtotalCouponDiscount, SubtotalLoyaltyPointDiscount, StatusDef
 Schema: ShopId, ShopMemberId, TradesGroupCode, TradesSlaveCode, OrderDateTime, OrderFinishDateTime, ChannelType, ChannelDetail, PaymentType, ShippingType, OuterProductSkuCode, ProductSkuCode, SalePageId, Qty, UnitPrice, SubtotalPrice, SubtotalSalesAmount, SubtotalPromotionDiscount, SubtotalCouponDiscount, SubtotalLoyaltyPointDiscount, StatusDef
Expected: SubtotalPrice but found: SubtotalSalesAmount
CSV file: file:///Users/jasonshen/Desktop/我的大蟒蛇/BDA_2025/91APP_Dataset(會員&主單&子單&商品頁&標籤)/Order_TS.csv


### 確認 SalePageId 全部都是缺失值

In [2]:
# Reassign from globals to local variables
df_member = globals()['df_member']
df_order_tg = globals()['df_order_tg']
df_order_ts = globals()['df_order_ts']
df_sale_page = globals()['df_sale_page']
df_segment = globals()['df_segment']


null_count = df_order_ts.filter(df_order_ts['SalePageId'].isNull()).count()
not_null_count = df_order_ts.filter(df_order_ts['SalePageId'].isNotNull()).count()
print(f"Null SalePageId count: {null_count}")
print(f"Not Null SalePageId count: {not_null_count}")

Null SalePageId count: 33963994
Not Null SalePageId count: 12560835


In [3]:
not_null = df_order_ts.filter(df_order_ts['SalePageId'].isNotNull())
not_null.show(5)

25/05/25 14:33:21 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ShopId, ShopMemberId, TradesGroupCode, TradesSlaveCode, OrderDateTime, OrderFinishDateTime, ChannelType, ChannelDetail, PaymentType, ShippingType, OuterProductSkuCode, ProductSkuCode, SalePageId, Qty, UnitPrice, SubtotalSalesAmount, SubtotalPrice, SubtotalPromotionDiscount, SubtotalCouponDiscount, SubtotalLoyaltyPointDiscount, StatusDef
 Schema: ShopId, ShopMemberId, TradesGroupCode, TradesSlaveCode, OrderDateTime, OrderFinishDateTime, ChannelType, ChannelDetail, PaymentType, ShippingType, OuterProductSkuCode, ProductSkuCode, SalePageId, Qty, UnitPrice, SubtotalPrice, SubtotalSalesAmount, SubtotalPromotionDiscount, SubtotalCouponDiscount, SubtotalLoyaltyPointDiscount, StatusDef
Expected: SubtotalPrice but found: SubtotalSalesAmount
CSV file: file:///Users/jasonshen/Desktop/我的大蟒蛇/BDA_2025/91APP_Dataset(會員&主單&子單&商品頁&標籤)/Order_TS.csv


+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+--------------+--------------+--------------------+--------------------+----------+---+---------+-------------+-------------------+-------------------------+----------------------+----------------------------+---------+
|ShopId|        ShopMemberId|     TradesGroupCode|     TradesSlaveCode|       OrderDateTime| OrderFinishDateTime| ChannelType|     ChannelDetail|   PaymentType|  ShippingType| OuterProductSkuCode|      ProductSkuCode|SalePageId|Qty|UnitPrice|SubtotalPrice|SubtotalSalesAmount|SubtotalPromotionDiscount|SubtotalCouponDiscount|SubtotalLoyaltyPointDiscount|StatusDef|
+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+--------------+--------------+--------------------+--------------------+----------+---+---------+-------------+--------------

In [4]:
not_null.printSchema()
not_null.select("SalePageId", "ProductSkuCode").show(5, truncate=False)


root
 |-- ShopId: integer (nullable = true)
 |-- ShopMemberId: string (nullable = true)
 |-- TradesGroupCode: string (nullable = true)
 |-- TradesSlaveCode: string (nullable = true)
 |-- OrderDateTime: string (nullable = true)
 |-- OrderFinishDateTime: string (nullable = true)
 |-- ChannelType: string (nullable = true)
 |-- ChannelDetail: string (nullable = true)
 |-- PaymentType: string (nullable = true)
 |-- ShippingType: string (nullable = true)
 |-- OuterProductSkuCode: string (nullable = true)
 |-- ProductSkuCode: string (nullable = true)
 |-- SalePageId: string (nullable = true)
 |-- Qty: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- SubtotalPrice: double (nullable = true)
 |-- SubtotalSalesAmount: double (nullable = true)
 |-- SubtotalPromotionDiscount: double (nullable = true)
 |-- SubtotalCouponDiscount: double (nullable = true)
 |-- SubtotalLoyaltyPointDiscount: double (nullable = true)
 |-- StatusDef: string (nullable = true)



+----------+------------------------+
|SalePageId|ProductSkuCode          |
+----------+------------------------+
|7440140.0 |AgrjMNVbKqPeRv46W/dS/Q==|
|7440140.0 |AgrjMNVbKqPeRv46W/dS/Q==|
|7403974.0 |c91k92jVrcbMzXKbj5UULg==|
|7403974.0 |c91k92jVrcbMzXKbj5UULg==|
|7089076.0 |XCmic6gofNxPjA5btXDwwg==|
+----------+------------------------+
only showing top 5 rows


In [6]:
df_sale_page_nn= df_sale_page.filter(df_sale_page['SalePageId'].isNotNull())
df_sale_page_nn.show(5)

+------+----------+--------------------------+---------------------------+
|ShopId|SalePageId|             SalePageTitle|SaleProductDescShortContent|
+------+----------+--------------------------+---------------------------+
|  NULL|   7440259|    DHC維他命D_30粒_30日份|              4511413615393|
|  NULL|   7440255| DHC綜合礦物質(30日份)90粒|              4511413609934|
|  NULL|   7440253|DHC精製魚油(DHA)(30日份...|              4511413602270|
|  NULL|   7440256|  DHC維他命B群(30日份)60粒|              4511413614860|
|  NULL|   7440257|DHC維他命B群(90日份)-180粒|              4511413404003|
+------+----------+--------------------------+---------------------------+
only showing top 5 rows


In [7]:
from pyspark.sql.functions import col

# Fix SalePageId: convert to int then string (optional depending on how df_sale_page looks)
not_null = not_null.withColumn("SalePageId", col("SalePageId").cast("double").cast("long").cast("string"))
not_null.show(5)
# df_sale_page = df_sale_page.withColumn("SalePageId", col("SalePageId").cast("long").cast("string"))
# df_valid_ts = df_order_ts.filter(df_order_ts['SalePageId'].isNotNull() & df_order_ts['ShopId'].isNotNull())

# df_joined = df_valid_ts.join(df_sale_page, on=["SalePageId"], how="left")

# df_joined.select("SalePageId").show(5)



25/05/25 14:35:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ShopId, ShopMemberId, TradesGroupCode, TradesSlaveCode, OrderDateTime, OrderFinishDateTime, ChannelType, ChannelDetail, PaymentType, ShippingType, OuterProductSkuCode, ProductSkuCode, SalePageId, Qty, UnitPrice, SubtotalSalesAmount, SubtotalPrice, SubtotalPromotionDiscount, SubtotalCouponDiscount, SubtotalLoyaltyPointDiscount, StatusDef
 Schema: ShopId, ShopMemberId, TradesGroupCode, TradesSlaveCode, OrderDateTime, OrderFinishDateTime, ChannelType, ChannelDetail, PaymentType, ShippingType, OuterProductSkuCode, ProductSkuCode, SalePageId, Qty, UnitPrice, SubtotalPrice, SubtotalSalesAmount, SubtotalPromotionDiscount, SubtotalCouponDiscount, SubtotalLoyaltyPointDiscount, StatusDef
Expected: SubtotalPrice but found: SubtotalSalesAmount
CSV file: file:///Users/jasonshen/Desktop/我的大蟒蛇/BDA_2025/91APP_Dataset(會員&主單&子單&商品頁&標籤)/Order_TS.csv


+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+--------------+--------------+--------------------+--------------------+----------+---+---------+-------------+-------------------+-------------------------+----------------------+----------------------------+---------+
|ShopId|        ShopMemberId|     TradesGroupCode|     TradesSlaveCode|       OrderDateTime| OrderFinishDateTime| ChannelType|     ChannelDetail|   PaymentType|  ShippingType| OuterProductSkuCode|      ProductSkuCode|SalePageId|Qty|UnitPrice|SubtotalPrice|SubtotalSalesAmount|SubtotalPromotionDiscount|SubtotalCouponDiscount|SubtotalLoyaltyPointDiscount|StatusDef|
+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+--------------+--------------+--------------------+--------------------+----------+---+---------+-------------+--------------

In [8]:
# df_valid_ts = df_order_ts.filter(df_order_ts['SalePageId'].isNotNull() & df_order_ts['ShopId'].isNotNull())

df_joined = not_null.join(df_sale_page_nn, on=["SalePageId"], how="left")

df_joined.select("SalePageId").show(5)

+----------+
|SalePageId|
+----------+
|   7440140|
|   7440140|
|   7403974|
|   7403974|
|   7089076|
+----------+
only showing top 5 rows


In [9]:
df_joined.show(10)

25/05/25 14:36:32 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ShopId, ShopMemberId, TradesGroupCode, TradesSlaveCode, OrderDateTime, OrderFinishDateTime, ChannelType, ChannelDetail, PaymentType, ShippingType, OuterProductSkuCode, ProductSkuCode, SalePageId, Qty, UnitPrice, SubtotalSalesAmount, SubtotalPrice, SubtotalPromotionDiscount, SubtotalCouponDiscount, SubtotalLoyaltyPointDiscount, StatusDef
 Schema: ShopId, ShopMemberId, TradesGroupCode, TradesSlaveCode, OrderDateTime, OrderFinishDateTime, ChannelType, ChannelDetail, PaymentType, ShippingType, OuterProductSkuCode, ProductSkuCode, SalePageId, Qty, UnitPrice, SubtotalPrice, SubtotalSalesAmount, SubtotalPromotionDiscount, SubtotalCouponDiscount, SubtotalLoyaltyPointDiscount, StatusDef
Expected: SubtotalPrice but found: SubtotalSalesAmount
CSV file: file:///Users/jasonshen/Desktop/我的大蟒蛇/BDA_2025/91APP_Dataset(會員&主單&子單&商品頁&標籤)/Order_TS.csv


+----------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+-----------+--------------+--------------------+--------------------+---+---------+-------------+-------------------+-------------------------+----------------------+----------------------------+---------+------+-----------------------------------+------------------------------------+
|SalePageId|ShopId|        ShopMemberId|     TradesGroupCode|     TradesSlaveCode|       OrderDateTime| OrderFinishDateTime| ChannelType|     ChannelDetail|PaymentType|  ShippingType| OuterProductSkuCode|      ProductSkuCode|Qty|UnitPrice|SubtotalPrice|SubtotalSalesAmount|SubtotalPromotionDiscount|SubtotalCouponDiscount|SubtotalLoyaltyPointDiscount|StatusDef|ShopId|                      SalePageTitle|         SaleProductDescShortContent|
+----------+------+--------------------+--------------------+--------------------+--------------------+-------------

In [10]:
from pyspark.sql.functions import col

df_2023_01 = df_joined.filter(
    (col("OrderDateTime") >= "2023-01-01") & (col("OrderDateTime") < "2023-02-01")
)

df_2023_01.show(5)
print(f"✅ 2023年1月資料筆數：{df_2023_01.count()}")


25/05/25 14:49:34 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ShopId, ShopMemberId, TradesGroupCode, TradesSlaveCode, OrderDateTime, OrderFinishDateTime, ChannelType, ChannelDetail, PaymentType, ShippingType, OuterProductSkuCode, ProductSkuCode, SalePageId, Qty, UnitPrice, SubtotalSalesAmount, SubtotalPrice, SubtotalPromotionDiscount, SubtotalCouponDiscount, SubtotalLoyaltyPointDiscount, StatusDef
 Schema: ShopId, ShopMemberId, TradesGroupCode, TradesSlaveCode, OrderDateTime, OrderFinishDateTime, ChannelType, ChannelDetail, PaymentType, ShippingType, OuterProductSkuCode, ProductSkuCode, SalePageId, Qty, UnitPrice, SubtotalPrice, SubtotalSalesAmount, SubtotalPromotionDiscount, SubtotalCouponDiscount, SubtotalLoyaltyPointDiscount, StatusDef
Expected: SubtotalPrice but found: SubtotalSalesAmount
CSV file: file:///Users/jasonshen/Desktop/我的大蟒蛇/BDA_2025/91APP_Dataset(會員&主單&子單&商品頁&標籤)/Order_TS.csv


+----------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+--------------+--------------+--------------------+--------------------+---+---------+-------------+-------------------+-------------------------+----------------------+----------------------------+---------+------+-----------------------------------+------------------------------------+
|SalePageId|ShopId|        ShopMemberId|     TradesGroupCode|     TradesSlaveCode|       OrderDateTime| OrderFinishDateTime| ChannelType|     ChannelDetail|   PaymentType|  ShippingType| OuterProductSkuCode|      ProductSkuCode|Qty|UnitPrice|SubtotalPrice|SubtotalSalesAmount|SubtotalPromotionDiscount|SubtotalCouponDiscount|SubtotalLoyaltyPointDiscount|StatusDef|ShopId|                      SalePageTitle|         SaleProductDescShortContent|
+----------+------+--------------------+--------------------+--------------------+--------------------+-------

✅ 2023年1月資料筆數：463800


In [11]:
from pyspark.sql.functions import sum as spark_sum

df_gmv_qty = df_2023_01.groupBy(
    "SalePageId", "SalePageTitle", "SaleProductDescShortContent"
).agg(
    spark_sum("SubtotalSalesAmount").alias("GMV"),
    spark_sum("Qty").alias("TotalQty")
)

df_gmv_qty.show(20)

25/05/25 14:54:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: OrderDateTime, SalePageId, Qty, SubtotalPrice
 Schema: OrderDateTime, SalePageId, Qty, SubtotalSalesAmount
Expected: SubtotalSalesAmount but found: SubtotalPrice
CSV file: file:///Users/jasonshen/Desktop/我的大蟒蛇/BDA_2025/91APP_Dataset(會員&主單&子單&商品頁&標籤)/Order_TS.csv


+----------+------------------------------------+----------------------------------------+---------+--------+
|SalePageId|                       SalePageTitle|             SaleProductDescShortContent|      GMV|TotalQty|
+----------+------------------------------------+----------------------------------------+---------+--------+
|   7534173|   【超值加價購】100%純棉植物纖維...|              ｜100%純棉植物纖維潔膚巾｜| 289377.0|    2923|
|   7507705|  【超值加價購】頂級山茶花黃金護唇膏|                ｜頂級山茶花黃金護唇膏｜| 349300.0|     700|
|   7044715|   凱婷 雙用立體眉彩筆W (銳角扁平芯)|                              持久不掉色|   6230.0|      18|
|   7061579|    SELENA五層可撕型敷面化妝棉80枚入|    100%棉，對肌膚溫和，使用後不殘留毛絮|   3027.0|      31|
|   8527306|       百特兔透明包包防塵袋-多款任選|   適用多型包款，防塵防髒保護，吊掛收...|    477.0|       3|
|   7069347|【即期品】好奇純水嬰兒濕巾加厚型2...|                  飲用級純水: 寶寶無負擔|  51246.0|    1314|
|   7550689|   【買2送1】76酵母胺基酸淨膚潔顏...|                ｜健康好膚質的保養捷徑｜|6348824.0|    3176|
|   7083786|         S.D.R.奢華白珍珠洗手乳500ml|含有保濕霜的特別配方，打造飽水柔嫩之肌底|   1490.0|      10|


In [15]:
df_top_1000 = df_gmv_qty.orderBy(
    col("GMV").desc(),
    col("TotalQty").desc()
).limit(1000)

df_top_1000.show(40, truncate=False)

25/05/25 15:08:21 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: OrderDateTime, SalePageId, Qty, SubtotalPrice
 Schema: OrderDateTime, SalePageId, Qty, SubtotalSalesAmount
Expected: SubtotalSalesAmount but found: SubtotalPrice
CSV file: file:///Users/jasonshen/Desktop/我的大蟒蛇/BDA_2025/91APP_Dataset(會員&主單&子單&商品頁&標籤)/Order_TS.csv


+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+--------+
|SalePageId|SalePageTitle                                                                                                                                                                                                                                     |SaleProductDescShortContent                                                                                                                                            |GMV        |TotalQty|
+----------+--------------------------------------------------------------------------------------------------

In [16]:
df_top_1000.coalesce(1).write \
    .option("header", True) \
    .csv("/Users/jasonshen/Desktop/我的大蟒蛇/BDA_2025/91APP_Dataset(會員&主單&子單&商品頁&標籤)/df_top_1000_gmv.csv")



25/05/25 15:12:14 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: OrderDateTime, SalePageId, Qty, SubtotalPrice
 Schema: OrderDateTime, SalePageId, Qty, SubtotalSalesAmount
Expected: SubtotalSalesAmount but found: SubtotalPrice
CSV file: file:///Users/jasonshen/Desktop/我的大蟒蛇/BDA_2025/91APP_Dataset(會員&主單&子單&商品頁&標籤)/Order_TS.csv


In [17]:
# Stop the Spark session
spark.stop()